# Multi-Agent Workflows

Environment setup and a custom Tavily web search tool for use by agents (OpenAI Agents SDK).

No agents are created in this notebook yet.

## 1. Environment setup

In [28]:
import os
from typing import TypedDict

import requests
from dotenv import load_dotenv
from agents import function_tool

# Load OPENAI_API_KEY and TAVILY_API_KEY from the existing .env file
load_dotenv(override=True)

for key in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    value = os.getenv(key)
    status = "set" if value and "here" not in value else "MISSING / placeholder"
    print(f"{key}: {status}")

OPENAI_API_KEY: set
TAVILY_API_KEY: set


## 2. Tavily web search tool

- Endpoint: `https://api.tavily.com/search` (POST, `Authorization: Bearer <key>`)
- Parameters: `query` and `max_results` (typed with `TypedDict`, default 2)

In [29]:
TAVILY_URL = "https://api.tavily.com/search"
DEFAULT_MAX_RESULTS = 2


class TavilySearchArgs(TypedDict):
    """Arguments for the Tavily web search."""

    query: str  # The search query
    max_results: int  # Number of results to return (use 2 unless told otherwise)


@function_tool
def tavily_search(args: TavilySearchArgs) -> str:
    """Search the web with Tavily and return titles, URLs and content snippets.

    Args:
        args: The search query and the maximum number of results (default 2).
    """
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        return "Error: TAVILY_API_KEY is not set in the environment."

    query = args["query"]
    max_results = args.get("max_results") or DEFAULT_MAX_RESULTS

    try:
        response = requests.post(
            TAVILY_URL,
            headers={"Authorization": f"Bearer {api_key}"},
            json={"query": query, "max_results": max_results},
            timeout=30,
        )
        response.raise_for_status()
    except requests.RequestException as exc:
        return f"Error: Tavily search failed: {exc}"

    results = response.json().get("results", [])
    if not results:
        return "No results found."

    return "\n\n".join(
        f"{i}. {r.get('title', '')}\n   URL: {r.get('url', '')}\n   {r.get('content', '')}"
        for i, r in enumerate(results, start=1)
    )

## 3. Quick test

Inspect the tool schema the agents will see, then invoke the tool the same way the SDK does.

In [30]:
import json

print(tavily_search.name)
print(json.dumps(tavily_search.params_json_schema, indent=2))

tavily_search
{
  "$defs": {
    "TavilySearchArgs": {
      "description": "Arguments for the Tavily web search.",
      "properties": {
        "query": {
          "title": "Query",
          "type": "string"
        },
        "max_results": {
          "title": "Max Results",
          "type": "integer"
        }
      },
      "required": [
        "query",
        "max_results"
      ],
      "title": "TavilySearchArgs",
      "type": "object",
      "additionalProperties": false
    }
  },
  "properties": {
    "args": {
      "description": "The search query and the maximum number of results (default 2).",
      "properties": {
        "query": {
          "title": "Query",
          "type": "string"
        },
        "max_results": {
          "title": "Max Results",
          "type": "integer"
        }
      },
      "required": [
        "query",
        "max_results"
      ],
      "title": "TavilySearchArgs",
      "type": "object",
      "additionalProperties": false
 

In [31]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

from agents.tool_context import ToolContext

payload = json.dumps({"args": {"query": "OpenAI Agents SDK handoffs", "max_results": 2}})
ctx = ToolContext(context=None, tool_name=tavily_search.name, tool_call_id="test-1", tool_arguments=payload)

# Run the async tool in its own thread/event loop (works in notebooks and plain Python)
with ThreadPoolExecutor(max_workers=1) as pool:
    result = pool.submit(asyncio.run, tavily_search.on_invoke_tool(ctx, payload)).result()
print(result)

1. Handoffs - OpenAI Agents SDK
   URL: https://openai.github.io/openai-agents-python/handoffs
   # Handoffs

Handoffs allow an agent to delegate tasks to another agent. This is particularly useful in scenarios where different agents specialize in distinct areas. For example, a customer support app might have agents that each specifically handle tasks like order status, refunds, FAQs, etc.

Handoffs are represented as tools to the LLM. So if there's a handoff to an agent named `Refund Agent`, the tool would be named `transfer_to_refund_agent`.

## Creating a handoff [...] You can create a handoff using the `handoff()` function provided by the Agents SDK. This function allows you to specify the agent to hand off to, along with optional overrides and input filters.

### Basic usage

Here's how you can create a simple handoff: [...] All agents have a `handoffs` param, which can either take an `Agent` directly, or a `Handoff` object that customizes the Handoff.

If you pass plain `Agent` i

In [32]:
import asyncio
import json
from concurrent.futures import ThreadPoolExecutor

from agents.tool_context import ToolContext


def run_tool(tool, query: str, max_results: int = 2) -> str:
    """Call a function tool directly (outside an agent) for testing."""
    payload = json.dumps({"args": {"query": query, "max_results": max_results}})
    ctx = ToolContext(
        context=None,
        tool_name=tool.name,
        tool_call_id="manual",
        tool_arguments=payload,
    )
    coro = tool.on_invoke_tool(ctx, payload)
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()


result = run_tool(tavily_search, "Latest news on quantum computing", max_results=2)
print(result)

1. Latest Quantum Computing News & Breakthroughs | Quantum Zeitgeist
   URL: https://quantumzeitgeist.com
   Quantum Hardware, Quantum Computing News

  ### Diraq Advances to Final Stage of Quantum Benchmarking Initiative

  by Rusty FlintOctober 8, 2026
 Xanadu wins IEEE synergy award for quantum community work

  Quantum Computing News, Quantum Ecosystem

  ### Xanadu wins IEEE synergy award for quantum community work

  by Rusty FlintOctober 7, 2026

 A bustling trading floor is filled with blurred figures and glowing screens displaying financial data in green, red, and white [...] Quantum Computing Business News, Quantum Companies

  ### Korea Quantum Computing to List on Nasdaq via Charlton Aria

  by Ivy DelaneyOctober 8, 2026
 A quantum computer in a glass case in a modern office

  Quantum Hardware, Quantum Funding

  ### Oratomic Says it secures $775 million to build fault-tolerant quantum computers

  by Rusty FlintOctober 8, 2026
 Universal Quantum secures over $100M to expa

## 4. Specialized agents

Two agents, each doing one job, sharing one Pydantic output type:

| Agent | Tools | Job | Output |
|-------|-------|-----|--------|
| **Researcher** | `tavily_search` | Search the web, return max 5 bullet points | `ResearchOutput` |
| **Analyst** | none | Turn research notes into trends, risks, insights (2 paragraphs) | `ResearchOutput` |

Both use `gpt-5-mini`.

In [33]:
from pydantic import BaseModel
from agents import Agent, Runner

MODEL = "gpt-5-mini"


class ResearchOutput(BaseModel):
    """Structured output passed between the agents."""

    summary: str


researcher = Agent(
    name="Researcher",
    model=MODEL,
    instructions=(
        "You are a web researcher. Use the tavily_search tool to find "
        "real-time, factual information about the user's topic. "
        "Call it with max_results=2 unless told otherwise. "
        "Return at most 5 concise bullet points of facts (one per line, "
        "starting with '- '). Include the source URL in a bullet when relevant. "
        "Do not analyze or speculate; only report what you found."
    ),
    tools=[tavily_search],
    output_type=ResearchOutput,
)

analyst = Agent(
    name="Analyst",
    model=MODEL,
    instructions=(
        "You are an analyst. You receive research notes and must extract the "
        "key trends, risks and insights from them. Write exactly 2 paragraphs. "
        "Use only the information in the notes; do not search the web."
    ),
    output_type=ResearchOutput,
)

print(researcher.name, "->", researcher.model, "| tools:", [t.name for t in researcher.tools])
print(analyst.name, "->", analyst.model, "| tools:", [t.name for t in analyst.tools])

Researcher -> gpt-5-mini | tools: ['tavily_search']
Analyst -> gpt-5-mini | tools: []


## 5. Test the Researcher and Analyst

`run_agent` runs the async `Runner.run` on one persistent background event loop, so it works in a notebook without `await` and can be called repeatedly.

In [34]:
import asyncio
import threading

# One persistent event loop in a background thread. Reusing it keeps the
# OpenAI client valid across runs (a new loop per call closes it -> errors).
_loop = asyncio.new_event_loop()
threading.Thread(target=_loop.run_forever, daemon=True).start()


def run_agent(agent, user_input: str):
    """Run an agent to completion and return the RunResult."""
    future = asyncio.run_coroutine_threadsafe(Runner.run(agent, user_input), _loop)
    return future.result()


QUERY = "Why is Labubu so popular and what are the rarest Labubu collectibles?"

research_result = run_agent(researcher, QUERY)
research = research_result.final_output  # a ResearchOutput instance

print(type(research).__name__)
print(research.summary)

ResearchOutput
- Labubu’s popularity is attributed to its “ugly‑cute” aesthetic, the blind‑box collecting mechanic, social‑media/celebrity influence, and collaborations/limited runs that create scarcity. (Source: https://www.demandsage.com/labubu-statistics)
- Labubu was created by Hong Kong illustrator Kasing Lung (introduced 2015) and gained wider recognition after a 2019 collaboration with Pop Mart; the line surged in popularity across East and Southeast Asia in 2024. (Source: https://en.wikipedia.org/wiki/Labubu)
- Rare Labubu figures (example: Chestnut Cocoa Labubu) have sold for over $149 on resale platforms; many Labubu listings on marketplaces sell above original retail due to limited stock. (Source: https://www.demandsage.com/labubu-statistics)
- Pop Mart’s Labubu lineup has expanded to hundreds of unique figures (over 300 in earlier reports), with frequent new releases and limited/collector editions driving rarity. (Source: https://www.demandsage.com/labubu-statistics)


In [35]:
# Pass the Researcher's notes to the Analyst
analysis_result = run_agent(analyst, research.summary)
analysis = analysis_result.final_output  # a ResearchOutput instance

print(type(analysis).__name__)
print(analysis.summary)

ResearchOutput
Labubu’s growth is driven by its “ugly‑cute” aesthetic, blind‑box collecting mechanic, social‑media and celebrity influence, and collaborations/limited runs that create scarcity. Created by Hong Kong illustrator Kasing Lung in 2015 and boosted by a 2019 collaboration with Pop Mart, the line surged in East and Southeast Asia in 2024; Pop Mart’s lineup has expanded to hundreds of unique figures (over 300 reported) with frequent new releases and limited/collector editions.
Scarcity and limited stock have produced notable resale premiums—rare figures (e.g., Chestnut Cocoa Labubu) have sold for over $149 and many marketplace listings trade above original retail. Insight/risk: the combination of limited runs and frequent drops sustains secondary‑market value and collector engagement, but the large number of unique releases could fragment the collector base and complicate long‑term valuation and completion for collectors.


## 6. Writer agent and the full pipeline

The **Writer** turns the query, research and analysis into an executive report. `manager_run()` chains
**Researcher → Analyst → Writer**, and all three share one `SQLiteSession` (in-memory) so each agent sees the earlier conversation.

In [36]:
import uuid

from pydantic import Field
from agents import SQLiteSession


class ReportOutput(BaseModel):
    """Final executive report produced by the Writer."""

    short_summary: str = Field(description="A 2-3 sentence executive summary.")
    markdown_report: str = Field(
        description="The full report in markdown, at least 500 words."
    )
    follow_up_questions: list[str] = Field(
        description="3 to 5 follow-up questions worth researching next."
    )


writer = Agent(
    name="Writer",
    model=MODEL,
    instructions=(
        "You are a senior report writer. You receive the original user question, "
        "the Researcher's notes and the Analyst's analysis. Write a polished "
        "executive report using only that material. Return: "
        "(1) short_summary: 2-3 sentences; "
        "(2) markdown_report: at least 500 words, with a title, headings "
        "(overview, key findings, trends, risks, conclusion) and the source URLs "
        "mentioned in the research; "
        "(3) follow_up_questions: 3 to 5 questions for further research."
    ),
    output_type=ReportOutput,
)


async def manager_run(user_query: str) -> ReportOutput:
    """Run Researcher -> Analyst -> Writer, sharing one SQLiteSession."""
    session = SQLiteSession(f"research-{uuid.uuid4().hex[:8]}")  # in-memory

    research = (await Runner.run(researcher, user_query, session=session)).final_output
    print("1/3 Researcher done")

    analysis = (await Runner.run(analyst, research.summary, session=session)).final_output
    print("2/3 Analyst done")

    writer_input = (
        f"User question:\n{user_query}\n\n"
        f"Research notes:\n{research.summary}\n\n"
        f"Analysis:\n{analysis.summary}"
    )
    report = (await Runner.run(writer, writer_input, session=session)).final_output
    print("3/3 Writer done")
    return report


def run_manager(user_query: str) -> ReportOutput:
    """Run manager_run on the persistent loop (no top-level await needed)."""
    return asyncio.run_coroutine_threadsafe(manager_run(user_query), _loop).result()


print(writer.name, "->", writer.model)

Writer -> gpt-5-mini


In [37]:
TEST_QUERY = "What is the public sentiment and expert reviews about the Tesla Cybertruck?"

final_report = run_manager(TEST_QUERY)

try:
    from IPython.display import Markdown, display

    display(Markdown("## Short summary\n\n" + final_report.short_summary))
    display(Markdown("## Full report\n\n" + final_report.markdown_report))
    questions = "\n".join(f"{i}. {q}" for i, q in enumerate(final_report.follow_up_questions, 1))
    display(Markdown("## Follow-up questions\n\n" + questions))
except ImportError:
    print(final_report.short_summary, final_report.markdown_report, sep="\n\n")
    print(*final_report.follow_up_questions, sep="\n")

print(f"\nReport length: {len(final_report.markdown_report.split())} words")

1/3 Researcher done
2/3 Analyst done
3/3 Writer done


## Short summary

Owner sentiment for the Tesla Cybertruck is strongly positive: Kelley Blue Book reports a 4.4/5 overall score, 4.6/5 reliability, and a 78% recommendation rate, with owners praising utility, performance, fit and finish, and technology. Expert reviews flag operational concerns—most notably maneuverability and rear visibility—reported by auto experts such as Sandy Munro and Edmunds’ Clint Simone (via Business Insider).

## Full report

# Tesla Cybertruck: Public Sentiment and Expert Reviews

## Overview

This report summarizes available owner sentiment and expert commentary on the Tesla Cybertruck, based on consumer reporting from Kelley Blue Book and expert coverage reported by Business Insider. Owner feedback is strongly positive on overall satisfaction and reliability, while automotive experts point to practical handling and visibility issues that may affect driver experience.

Source URLs

- https://www.kbb.com/tesla/cybertruck
- https://www.businessinsider.com/tesla-cybertruck-can-be-difficult-to-maneuver-auto-expert-says-2024-3

## Key findings

- Strong owner satisfaction: Kelley Blue Book (KBB) reports that 2027 Tesla Cybertruck owners give the vehicle an overall rating of 4.4 out of 5, a reliability rating of 4.6 out of 5, and a 78% recommendation rate. These metrics indicate that early or current owners regard the Cybertruck as above average for its class in quality and dependability.

- Positive owner commentary: KBB owner comments emphasize a combination of utility and performance. Common owner sentiments cited include the Cybertruck’s "utility of a truck, performance of a sports car, luxury fit and finish, and excellent technology." Representative quotes in KBB reporting include "Exceeds expectations" and "Best vehicle I have owned."

- Expert concerns about maneuverability and visibility: Automotive experts — highlighted in Business Insider’s coverage — raise operational issues. Sandy Munro and other auto experts observed that the Cybertruck can be difficult to maneuver, particularly in reversing or tight spaces. Edmunds’ automotive journalist Clint Simone noted limited rear visibility: when the cargo bed cover is deployed, "You can't see a darn thing out of the back," and drivers often rely on a continuous camera feed displayed on the center screen.

- Real-world visibility issues corroborated publicly: The Business Insider piece references public incidents and demonstrations (e.g., celebrities having difficulty handling the vehicle) that mirror expert critiques, indicating the visibility/maneuverability issues are noticeable outside controlled tests.

## Trends

- Owner satisfaction vs. expert scrutiny: There is a clear divergence between owner satisfaction metrics and expert operational critiques. Owners emphasize experiential positives (utility, performance, tech, reliability), while experts focus on objective handling and safety/user-interface concerns.

- Acceptance of camera-based visibility: Experts note that rear visibility relies on continuous camera feeds rather than traditional sightlines, and some drivers continue to habitually check mirrors despite the camera feed. This suggests an ongoing adaptation period where drivers adjust to different visibility systems; acceptance appears mixed.

- High praise for technology and performance: Owners’ praise for technology and performance underscores Tesla’s success in delivering perceived value from EV powertrain, software features, and a distinctive design package, which may offset practical critiques for many buyers.

## Risks

- Maneuverability and blind spots: Expert reports identify difficulty maneuvering and poor rear sightlines (especially with the bed cover deployed) as practical risks that can affect everyday usability and may increase the likelihood of minor collisions or driver discomfort in low-speed operations.

- Dependence on camera systems: The vehicle’s reliance on camera feeds for rear visibility introduces risk if cameras are obscured, fail, or if drivers are not fully comfortable or accustomed to using camera-based views instead of direct mirror sightlines.

- Potential for public perception swings: While current owner sentiment is positive, continued high-profile incidents or persistent negative expert coverage of practical driving issues could affect broader public perception and prospective buyer confidence.

## Conclusion

The Tesla Cybertruck receives strong marks from current owners for overall satisfaction and reliability, with many praising its combination of truck utility, sports-car performance, luxury fit and finish, and technology. However, automotive experts have raised consistent, practical concerns about maneuverability and rear visibility, emphasizing reliance on camera feeds and the awkwardness some drivers experience when reversing or parking. For stakeholders, the Cybertruck’s market position appears supported by enthusiastic owners and strong perceived capabilities, but the vehicle’s everyday usability in tight or low-visibility scenarios remains a focal point for expert criticism and a potential area for further design or software refinements.

Sources

- Kelley Blue Book — 2027 Tesla Cybertruck: https://www.kbb.com/tesla/cybertruck
- Business Insider — "Tesla Cybertruck Can Be Difficult to Maneuver, Auto Expert Says": https://www.businessinsider.com/tesla-cybertruck-can-be-difficult-to-maneuver-auto-expert-says-2024-3


## Follow-up questions

1. How widespread are visibility and maneuverability complaints among a larger sample of expert reviews (e.g., Edmunds, Car and Driver, MotorTrend)?
2. Have there been software or hardware updates (OTA or recalls) from Tesla addressing rear visibility or camera assist for the Cybertruck?
3. How do accident or insurance claim rates for the Cybertruck compare to other pickup trucks in similar classes?
4. What proportion of prospective buyers cite seen expert critiques (visibility/maneuverability) as a barrier to purchase?
5. How do owner satisfaction and reliability scores evolve over the next 12–24 months as more owners accumulate real-world experience?


Report length: 663 words


## 2. Tavily web search tool

- Endpoint: `https://api.tavily.com/search` (POST, `Authorization: Bearer <key>`)
- Parameters: `query` and `max_results` (typed with `TypedDict`, default 2)